# Model training

Trains and — more importantly — honestly evaluates a per-pixel classifier for waterhole
surface state.

## Read this before reading any score

**Your effective sample size is the number of labelled sites, not pixels.** The table below
has ~32,000 rows, but pixels within a tile are almost perfectly autocorrelated: knowing one
tells you most of what its neighbours will say. With 9 labelled sites, the honest n is 9.

Every split in this notebook therefore holds out **whole waterholes**. `wh_train` has no
code path for a random pixel-level split — it does not import `train_test_split` or `KFold`
at all, and its only splitter raises if handed fewer than 3 sites. A random split here would
report something like 0.95 and mean nothing.

Expect the flexible models to do *worse* than the simple one. That is not a bug; it is what
overfitting to 9 sites looks like.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "cookie-cutting" else Path.cwd() / "cookie-cutting"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import numpy as np
import pandas as pd

import wh_config
import wh_features
import wh_inventory
import wh_train

pd.set_option("display.width", 140)

cfg = wh_config.load()
manifest = wh_inventory.load_manifest(cfg)
print("config", cfg.source_path.name, "hash", cfg.hash)

## Parameters

In [ ]:
PARAMS = wh_features.FeatureParams(
    # 60 m atmospheric bands B1 and B9 excluded: resampled to 10 m they carry no
    # surface information worth having.
    reflectance_bands=("B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"),

    indices=("mndwi", "ndwi", "ndvi", "ndti", "ndmi", "nd_rededge",
             "red_green_ratio", "awei_sh", "awei_nsh"),

    # Windowed mean and SD, so a pixel knows whether it sits in a uniform patch
    # or on an edge. NaN-aware: a window overlapping a cloud gap averages the
    # pixels that were observed, not zeros.
    context_windows=(3, 9),
    context_indices=("mndwi", "ndvi"),

    # Indices given the full temporal treatment. NDTI carries turbidity and NDMI
    # responds to water UNDER a canopy — the case MNDWI gets wrong when sedges
    # cover a waterhole.
    temporal_indices=("mndwi", "ndvi", "ndti", "ndmi"),
)

INCLUDE_PSEUDO = False    # run pseudo_labelling.ipynb first, then try True
REBUILD_TABLE = True      # False reloads the cached table from derived/

PARAMS

## Build the training table

One row per labelled pixel, carrying `site_id` and `year_month` alongside every feature.
Those two columns are what make a grouped split possible, so they travel with every row.

Sites are processed one at a time so each site's temporal features are computed once rather
than once per labelled month. Expect a couple of minutes.

In [ ]:
if REBUILD_TABLE:
    table = wh_train.build_training_table(manifest, cfg, PARAMS, include_pseudo=INCLUDE_PSEUDO)
    wh_train.save_table(table, cfg)
else:
    table = wh_train.load_table(cfg)

features = wh_features.feature_columns(table)
print(f"\n{len(table):,} rows x {len(features)} features")
print(f"{table['site_id'].nunique()} sites, {table['year_month'].nunique()} months")
print(f"sources: {table['source'].value_counts().to_dict()}")

### Class coverage — the table that predicts what will fail

Read the **sites** column, not the pixels column. A class present at only 2 sites will be
absent from training entirely in some folds, and cannot be expected to work.

In [ ]:
coverage = table.groupby("class_id").agg(
    pixels=("row", "size"),
    tiles=("year_month", "nunique"),
    sites=("site_id", "nunique"),
)
coverage.insert(0, "class", [cfg.class_by_id(i).name for i in coverage.index])
coverage = coverage.sort_values("sites")

thin = coverage[coverage["sites"] < 3]
if len(thin):
    print("classes at fewer than 3 sites — expect these to fail, and not because "
          "the model is bad:")
    print("  " + ", ".join(thin["class"]))
coverage

In [ ]:
# Features that are mostly NaN mean too few pixels had enough observed months
# to fit — a data problem, not something for the imputer to paper over.
missing = wh_features.describe_missing(table, threshold=0.2)
print(f"{len(missing)} feature(s) more than 20% NaN")
missing.head(10)

## Cross-validation

Leave-one-site-out: 9 folds, each holding out one whole waterhole. The per-fold line shows
which sites the model can and cannot generalise to.

In [ ]:
results = {}
for model_name in ("logistic_regression", "random_forest", "gradient_boosting"):
    print(f"\n=== {model_name} ===")
    results[model_name] = wh_train.cross_validate(table, cfg, model_name)
    print(results[model_name].summary())

print("\n" + "=" * 60)
pd.DataFrame([
    {"model": name, "macro_f1": ev.macro_f1, "weighted_f1": ev.weighted_f1}
    for name, ev in results.items()
]).set_index("model").round(3)

**Logistic regression is a diagnostic, not a contender.** If it matches or beats the
trees, the classes are close to linearly separable in this feature space *and* the trees are
overfitting the handful of sites they were given. Both are worth knowing before adding more
features.

In [ ]:
BEST = "gradient_boosting"
evaluation = results[BEST]

print("per class:")
print(evaluation.per_class.round(3).to_string())
print("\nmacro F1 weights every class equally, so the rare broken ones drag it down;")
print("weighted F1 is dominated by surrounding vegetation and dry bare ground.")

In [ ]:
print("per site (the table that matters):")
evaluation.per_site.round(3)

In [ ]:
print("confusion matrix — read the rows to see what each true class gets mistaken for:")
evaluation.confusion

## Ablation: does the temporal design earn its keep?

The central claim of this pipeline is that normalising a pixel against its own history makes
the ambiguous dry-season months separable. `instantaneous_only` tests that directly — it is
the same model with the temporal columns removed.

**Already measured, and the config now reflects the answer.** Leave-one-site-out over the 9
labelled sites:

| feature set | features | macro F1 |
|---|---|---|
| instantaneous_only | 28 | 0.358 |
| all_features (harmonic on) | 93 | 0.433 |
| no_harmonic | 61 | 0.489 |
| model_free_temporal | 57 | **0.497** |

Temporal self-normalisation is worth **+0.14 macro F1**, a 39% relative gain — the design's
central claim holds. But the harmonic block *hurt*, so `features.temporal.harmonic_enabled`
is now `false`.

Because of that, the top three rows below are identical by construction: with the harmonic
off there is nothing left to ablate from it. Set `harmonic_enabled: true` in the config and
rebuild the table to reproduce the comparison.

In [ ]:
ablation = wh_train.run_ablation(table, cfg, PARAMS, model_name=BEST)
print()
ablation.round(4)

## Temporal holdout

Reported for completeness and **known to be optimistically biased**: a held-out month's
temporal features were computed from that pixel's whole history, training months included.
Removing that leak would mean recomputing every temporal feature per fold from training
months only.

The grouped-site CV above has no equivalent problem — holding out a site holds out its
history too. Trust that number, not this one.

In [ ]:
train_index, test_index = wh_train.temporal_holdout_split(
    table, cfg["training"]["cv"]["temporal_holdout_months"]
)
model = wh_train.make_model(BEST, cfg)
model.fit(table.iloc[train_index][features], table.iloc[train_index]["class_id"])
predicted = model.predict(table.iloc[test_index][features])

holdout = wh_train.evaluate_predictions(
    table.iloc[test_index]["class_id"].to_numpy(), predicted,
    table.iloc[test_index]["site_id"].to_numpy(), cfg, BEST, "temporal_holdout",
)
print(holdout.summary(), "  <- optimistically biased, see above")
holdout.per_class.round(3)[["f1", "iou", "support"]]

## Fit on everything and persist

The final model is fitted on all labelled sites. Its recorded CV score comes from the
grouped cross-validation above, not from this fit — a model scored on its own training data
is meaningless.

The feature list and config hash are saved beside it, because a model applied with features
in a different order produces confident nonsense rather than an error.

In [ ]:
final_model = wh_train.make_model(BEST, cfg)
final_model.fit(table[features], table["class_id"])

model_path, manifest_path = wh_train.save_model(
    final_model, features, cfg, PARAMS, results[BEST], table
)
print(f"model    -> {model_path}")
print(f"manifest -> {manifest_path}")

In [ ]:
# Reload and confirm it round-trips.
reloaded, meta = wh_train.load_model(cfg)
print(f"{meta['model_class']}, {meta['n_features']} features, "
      f"CV macro F1 {meta['cv_macro_f1']:.3f}")
assert list(meta["feature_names"]) == list(features)
assert np.array_equal(reloaded.predict(table[features].head(100)),
                      final_model.predict(table[features].head(100)))
print("round-trip OK")

## Where to spend the next labelling session

Ranked by what would most improve the model: classes at fewer than 3 sites cannot be
cross-validated at all, and sites with low per-site F1 are where generalisation is failing.

In [ ]:
print("classes needing more SITES (not more pixels):")
print(coverage[coverage["sites"] < 4][["class", "pixels", "sites"]].to_string(index=False))

print("\nworst-performing sites — look at these in the labeller and check the labels "
      "are right before blaming the model:")
print(evaluation.per_site.head(3).round(3).to_string())